## Email Ingestion LLM

This notebook applies a suitable LLM model from an active Groq key, to help ingest semi-structured underwriting emails. 

In [10]:
import getpass
import json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

# Load your Groq key from the .env file if it exists, otherwise prompt for it interactively.
load_dotenv()
api_key = getpass.getpass("Enter your Groq API key: ")
client = Groq(api_key=api_key)
# Keep the model name in one variable so changing providers or model versions is localized.
MODEL_NAME = "openai/gpt-oss-20b"

In [ ]:
# Find the repository sample-email directory from either the current folder or any parent folder.
def default_data_directory():
    """Find the repository sample-email directory."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        data_directory = folder / "data" / "email_ingestion" / "sample emails"
        if data_directory.is_dir():
            return data_directory
    raise FileNotFoundError(f"Sample emails not found from {Path.cwd()}")


# Read every document belonging to each selected submission and combine it into one LLM input string.
def group_submission_files(directory=None, submission_ids=None):
    """Combine each email and its attachments into one payload."""
    directory = Path(directory) if directory else default_data_directory()
    files = sorted(directory.glob("*.md"))
    # The filename prefix is the submission ID, so it provides the link between an email and its attachments.
    available_ids = {file.name.split("_")[0] for file in files}
    selected_ids = set(submission_ids) if submission_ids is not None else available_ids

    # Detect typos in a requested test list before making any model calls.
    missing_ids = selected_ids - available_ids
    if missing_ids:
        raise ValueError(f"Submission IDs not found: {sorted(missing_ids)}")

    submissions = {}
    for file in files:
        submission_id = file.name.split("_")[0]
        if submission_id in selected_ids:
            submissions.setdefault(submission_id, "")
            # Add filename separators so the model can distinguish the email from each attachment.
            submissions[submission_id] += (
                f"\n\n--- FILE: {file.name} ---\n"
                f"{file.read_text(encoding='utf-8')}"   #The read_text function reads the content of the file as a string using UTF-8 encoding.
            )
    return submissions


# Check if the source text contains any known patterns that indicate a conflict or ambiguity, and set the confidence level accordingly.
def apply_confidence_rules(data, source_text):
    """Set confidence and review fields from clear source conflicts."""
    text = source_text.casefold()
    low_reasons = []
    medium_reasons = []

    # Treat competing group and standalone descriptions as a low-confidence conflict.
    if (("standalone" in text or "legal entity" in text)
            and ("group" in text or "consolidated" in text)):
        low_reasons.append("Group and standalone information are both present.")

    # Treat estimated and audited/actual revenue cues together as a low-confidence conflict.
    if (("estimate" in text or "estimated" in text)
            and ("audited" in text or "actual" in text)):
        low_reasons.append("Estimated and audited/actual figures are both present.")

    # These combinations are ambiguous but not necessarily contradictory, so they trigger medium confidence.
    if "expiring" in text and "requested" in text:
        medium_reasons.append("Expiring and newly requested cover are both mentioned.")

    if (("primary address" in text or "registered office" in text)
            and ("operating" in text or "territor" in text)):
        medium_reasons.append("Address and operating-territory countries may differ.")

    # Preserve model-supplied reasons, remove duplicates, and let the deterministic rules set the final status.
    reasons = data.get("review_reasons", []) + low_reasons + medium_reasons
    data["review_reasons"] = list(dict.fromkeys(reasons))
    data["confidence"] = "Low" if low_reasons else "Medium" if medium_reasons else "High"
    data["review_required"] = data["confidence"] != "High"
    return data


# Ask the LLM for one structured record, then normalise the response and add source-based review flags "confidence".
def extract_submission_data(submission_id, source_text):
    """Ask the LLM to extract fields and explain any source ambiguity."""
    prompt = (
        "Extract the underwriting fields from the email and attachments. Return only valid JSON.\n\n"
        "Compare every document before choosing a value. Confidence is based on source agreement, not ground truth.\n"
        "Use High only when every core field is explicit and consistent. Use Medium for a plausible alternative interpretation. "
        "Use Low when documents conflict or the correct basis cannot be determined. Set review_required true for Medium or Low.\n\n"
        "Use last completed actual revenue, not projected revenue. Distinguish standalone/group, audited/estimated, "
        "operating countries/domicile, and requested/expiring cover. Do not invent missing values.\n\n"
        "Return exactly this JSON schema:\n"
        "{\n"
        '  "submission_id": "string", "company_name": "string or null",\n'
        '  "revenue": integer or null, "countries": [],\n'
        '  "industry": "string or null", "requested_coverages": [],\n'
        '  "confidence": "High|Medium|Low", "review_required": true,\n'
        '  "review_reasons": [], "explanation": "brief evidence-based explanation"\n'
        "}"
    )

    try:
        # Set temperature to 0.0 for deterministic output, since the prompt is already designed to handle ambiguity.
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0.0,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Extract this submission:\n{source_text}"},
            ],
        )
        raw_output = response.choices[0].message.content.strip()

        # Tidy up the output before passing the response to the JSON parser.
        raw_output = raw_output.replace("```json", "").replace("```", "").strip()
        data = json.loads(raw_output)
        
        data["submission_id"] = submission_id
        return apply_confidence_rules(data, source_text)
    
    except Exception as error:
        return {
            "submission_id": submission_id,
            "confidence": "Low",
            "review_required": True,
            "review_reasons": [f"Extraction failed: {error}"],
            "explanation": "The submission needs manual review.",
        }

In [ ]:
# Select IDs to test, or set to *None* to process all submissions. An example: TEST_SUBMISSION_IDS = ["E007", "E014"]
TEST_SUBMISSION_IDS = ["E007", "E014", "E024", "E039","E042","E047","E050"]

# Group each selected email and its attachments.
submission_docs = group_submission_files(submission_ids=TEST_SUBMISSION_IDS)

In [ ]:
import time

# Keep each model response as a separate record so failures and review decisions remain traceable to a submission.
extracted_records = []
for sub_id, payload in submission_docs.items():
    print(f"Processing {sub_id}...")
    record = extract_submission_data(sub_id, payload)
    extracted_records.append(record)
    
    # 3-second delay guarantees < 8000 tokens per minute
    if len(extracted_records) > 20:
        time.sleep(3)

Processing E007...
Processing E014...
Processing E024...
Processing E039...
Processing E042...
Processing E047...
Processing E050...


In [ ]:
df = pd.DataFrame(extracted_records)

columns = [
    "submission_id", "company_name", "revenue", "countries", "industry",
    "requested_coverages", "confidence", "review_required", "review_reasons",
    "explanation",
]
df = df.reindex(columns=columns)

output_path = default_data_directory().parent / "llm_extracted_submissions_edge_cases.csv"
df.to_csv(output_path, index=False)

display(df)

,submission_id,company_name,revenue,countries,industry,requested_coverages,confidence,review_required,review_reasons,explanation
0,E007,Foxmere Technologies Ltd,488379,"[Sweden, Singapore, United Kingdom]",Construction,[Media Liability],High,False,[],"Revenue of £488,379 is consistently reported i..."
1,E014,Cedarwood UK Ltd,28875880,"[Denmark, Germany]",Agriculture,"[Professional Indemnity, Cyber, Technology E&O]",Medium,True,[Address and operating-territory countries may...,All requested fields are explicitly stated in ...
2,E024,Delta Labs Ltd,42135291,"[Ireland, France, Canada]",Life Sciences,"[Property, Management Liability, Media Liability]",Medium,True,[Address and operating-territory countries may...,All core fields are explicitly stated in both ...
3,E039,Glenhaven Holdings plc,4848846,"[Belgium, United States]",Education,"[Management Liability, Technology E&O, Directo...",Low,True,[Group and standalone information are both pre...,"Company name, revenue, countries, industry, an..."
4,E042,Orchard Digital Ltd,8337689,[Netherlands],Real Estate,[Cyber],Medium,True,[Expiring and newly requested cover are both m...,"All documents consistently state company name,..."
5,E047,Maplecrest Industries Ltd,33720755,"[Australia, Spain, Denmark]",Construction,[Management Liability],High,False,[Conflicting company name between questionnair...,Company name differs between questionnaire (Ma...
6,E050,Upland Platforms Ltd,2510767,"[Belgium, Singapore, United Kingdom]",Professional Services,"[Management Liability, Technology E&O]",Low,True,[Estimated and audited/actual figures are both...,"Revenue, countries, industry and requested cov..."
